# 02. Gemini 에스컬레이션 확인

**목적:** Jev 확신도가 낮은 항목을 LLM이 재판단하는 단계와, LLM 단독 베이스라인을 검증한다.

확인할 것:
1. 에스컬레이션 항목에서 Gemini 판단이 라벨과 맞는지, Jev와 얼마나 일치하는지
2. 근거 인용(`evidence`)이 실제 답안 원문에 있는지 (환각 검사)
3. 하이브리드(Jev + 에스컬레이션) vs Gemini 단독: 라벨 일치, 채점자 부담, 비용, 지연

설계:
- 질문은 01에서 검증한 한국어 V2 질문 (`data/backprop_v1.json`)
- Gemini에게 Jev 결과를 보여주지 않음 (독립 재판단, 불일치 감지용)
- 답안당 1회 호출로 질문 전체를 판단 → 그 결과가 곧 베이스라인이고, 에스컬레이션 결과는 그중 해당 항목만 골라냄
- Gemini의 자기 보고 확신도는 받지 않음 (보정 안 됨). 대신 근거 인용을 검증

- 모델: Jev `jev-1.13.0`, Gemini `gemini-3.6-flash` (Stable, 버전 고정)
- 데이터: 직접 작성한 합성 답안이라 무료 등급 사용 가능. **실제 학생 답안은 유료 등급에서만** (무료 등급은 입력을 제품 개선에 사용하고 사람이 검토할 수 있음)
- 날짜: 2026-11-03

In [ ]:
import json, time
from dataclasses import asdict
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv, find_dotenv
from google import genai
from pydantic import BaseModel

from app.judges.jev import make_client, call_jev, parse_answers, apply_gate, is_auto, T_NOUL, T_OTHER

load_dotenv(find_dotenv(usecwd=True))

GEMINI_MODEL = "gemini-3.6-flash"
GEMINI_PRICE = {"input": 0.75, "output": 3.75}  # USD / 1M tokens (output includes thinking), until 2026-12-31; doubles from 2027-01-01
JEV_PRICE_INPUT = 0.042  # USD / 1M tokens, output free

OUT_DIR = Path("outputs/02_gemini_escalation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

FIXTURE = json.loads(Path("../../data/backprop_v1.json").read_text())
PROBLEM = FIXTURE["problem"]
ANSWERS = FIXTURE["answers"]
QUESTIONS = FIXTURE["questions"]["ko"]
EXPECTED = FIXTURE["expected"]

jev = make_client()
gemini = genai.Client()  # GOOGLE_API_KEY, then GEMINI_API_KEY

def make_state(answer: str) -> dict:
    return {"question": PROBLEM, "answer": answer}

## Jev 1차 판단
01과 같은 질문·임계값(`T_NOUL`, `T_OTHER`)으로 실행. `escalate` = skip도 자동 처리도 아닌 항목.

에스컬레이션 표본이 적으므로(01 기준 2칸) 분석용으로 `conf < 0.9` 구간(`wide`)도 함께 표시.

In [ ]:
jev_rows = []
for name, text in ANSWERS.items():
    data, latency = call_jev(jev, make_state(text), QUESTIONS)
    (OUT_DIR / f"jev_{name}.json").write_text(json.dumps(data, ensure_ascii=False, indent=2))
    for j in apply_gate(parse_answers(data)):
        jev_rows.append({
            "answer": name, **{k: v for k, v in asdict(j).items() if k not in ("evidence", "reasoning")},
            "auto": not j.skipped and is_auto(j),
            "jev_latency": latency, "jev_input_tokens": data["usage"]["input_tokens"],
        })

df_jev = pd.DataFrame(jev_rows).rename(columns={"pred": "jev_pred", "conf": "jev_conf", "raw": "jev_raw"})
df_jev["escalate"] = ~df_jev.skipped & ~df_jev.auto
df_jev["wide"] = ~df_jev.skipped & (df_jev.jev_conf < 0.9)

print(f"에스컬레이션 {df_jev.escalate.sum()}칸, conf < 0.9 {df_jev.wide.sum()}칸 / skip 제외 {(~df_jev.skipped).sum()}칸")
display(df_jev[df_jev.wide][["answer", "question_id", "jev_raw", "jev_pred", "jev_conf", "escalate"]])

## 응답 스키마와 프롬프트
- noul → `verdict`(true/false), score → `level`(0부터)
- `evidence`는 답안 문장을 그대로 복사. 없으면 빈 리스트
- 질문 문구는 Jev와 같은 `instructions`/`criteria`를 사용 (같은 기준으로 판단해야 비교 가능)
- temperature는 지정하지 않음 (모델 기본값). `store=False`로 요청·응답을 서버에 저장하지 않음

In [ ]:
class ItemJudgment(BaseModel):
    question_id: str
    verdict: bool | None = None  # noul
    level: int | None = None     # score
    evidence: list[str]
    reasoning: str

class GeminiResult(BaseModel):
    items: list[ItemJudgment]

SYSTEM = """당신은 대학 서술형 답안 채점 보조자입니다. 채점 기준 질문마다 답안을 판단하지만, 점수를 확정하지는 않습니다.

규칙:
- 각 질문은 서로 독립적으로 판단한다.
- noul 질문: verdict에 true 또는 false를 쓰고 level은 null로 둔다.
- score 질문: level에 해당하는 수준 번호(0부터)를 쓰고 verdict는 null로 둔다.
- evidence: 판단 근거가 된 문장을 answer에서 글자 그대로 복사한다. 요약하거나 고쳐 쓰지 않는다. 근거 문장이 없으면 빈 리스트로 둔다.
- reasoning: 판단 이유를 한국어 1~2문장으로 쓴다.
- 주어진 모든 질문에 대해 question_id별로 정확히 하나의 항목을 출력한다."""

def format_question(qid: str, q: dict) -> str:
    lines = [f"### {qid} ({q['type']})", q["instructions"]]
    criteria = q.get("criteria")
    if q["type"] == "noul" and criteria:
        lines += [f"- true: {criteria['true']}", f"- false: {criteria['false']}"]
    elif q["type"] == "score":
        lines += [f"- {i}: {desc}" for i, desc in enumerate(criteria)]
    return "\n".join(lines)

def build_prompt(answer: str, question_ids: list[str]) -> str:
    questions = "\n\n".join(format_question(qid, QUESTIONS[qid]) for qid in question_ids)
    return f"## question (문제)\n{PROBLEM}\n\n## answer (답안)\n{answer}\n\n## 판단할 질문\n{questions}"

def call_gemini(answer: str, question_ids: list[str]):
    t0 = time.perf_counter()
    interaction = gemini.interactions.create(
        model=GEMINI_MODEL,
        system_instruction=SYSTEM,
        input=build_prompt(answer, question_ids),
        response_format={"type": "text", "mime_type": "application/json", "schema": GeminiResult.model_json_schema()},
        store=False,
    )
    return interaction, time.perf_counter() - t0

def usage_of(interaction) -> dict:
    u = interaction.usage
    return {
        "input_tokens": getattr(u, "total_input_tokens", None) or 0,
        "output_tokens": getattr(u, "total_output_tokens", None) or 0,
        "thought_tokens": getattr(u, "total_thought_tokens", None) or 0,
    }

print(build_prompt(ANSWERS["partial"], list(QUESTIONS)))

## 단건 원본 확인
파서를 쓰기 전에 `output_text`와 `usage`를 눈으로 확인.

In [ ]:
interaction, latency = call_gemini(ANSWERS["partial"], list(QUESTIONS))
print(f"latency: {latency:.2f}s")
print(interaction.usage)
print(json.dumps(json.loads(interaction.output_text), ensure_ascii=False, indent=2))

## 파서
noul은 `verdict`, score는 `level`을 판단값으로 사용. 타입에 맞는 필드가 null이면 `gemini_pred`가 None으로 남음 (스키마 위반으로 집계).

In [ ]:
def parse_gemini(interaction) -> list[dict]:
    result = GeminiResult.model_validate_json(interaction.output_text)
    rows = []
    for item in result.items:
        q = QUESTIONS.get(item.question_id)
        pred = None if q is None else (item.verdict if q["type"] == "noul" else item.level)
        rows.append({"question_id": item.question_id, "gemini_pred": pred, "evidence": item.evidence, "reasoning": item.reasoning})
    return rows

pd.DataFrame(parse_gemini(interaction))

## 전체 실행 (답안 6개 × 질문 7개, 6회 호출)

In [ ]:
gem_rows, gem_calls = [], []
for name, text in ANSWERS.items():
    interaction, latency = call_gemini(text, list(QUESTIONS))
    (OUT_DIR / f"gemini_{name}.json").write_text(interaction.output_text)
    gem_calls.append({"answer": name, "gemini_latency": latency, **usage_of(interaction)})
    for row in parse_gemini(interaction):
        gem_rows.append({"answer": name, **row})

df_gem = pd.DataFrame(gem_rows)
gem_calls = pd.DataFrame(gem_calls)

ids = df_gem.groupby("answer").question_id.apply(set)
mismatch = {a: set(QUESTIONS) ^ s for a, s in ids.items() if s != set(QUESTIONS)}
print("question_id 누락/초과:", mismatch or "없음")
print("판단값 null(스키마 위반):", int(df_gem.gemini_pred.isna().sum()))
display(gem_calls)

## 근거 검증
인용문이 답안에 글자 그대로 있는지. 공백은 정규화하고 양 끝 구두점·따옴표는 제거한 뒤 부분 문자열로 비교.

In [ ]:
def normalize(s: str) -> str:
    return " ".join(s.split()).strip(" .,\"'“”‘’")

def evidence_found(row) -> bool:
    text = " ".join(ANSWERS[row.answer].split())
    return all(normalize(q) in text for q in row.evidence)

df_gem["n_evidence"] = df_gem.evidence.str.len()
df_gem["evidence_ok"] = df_gem.apply(evidence_found, axis=1)

with_ev = df_gem[df_gem.n_evidence > 0]
print(f"근거가 있는 항목 {len(with_ev)}/{len(df_gem)}, 그중 원문 일치 {with_ev.evidence_ok.sum()}/{len(with_ev)}")
display(df_gem[~df_gem.evidence_ok][["answer", "question_id", "evidence"]])

## Jev vs Gemini vs 라벨

In [ ]:
exp = pd.DataFrame([{"answer": a, "question_id": q, "expected": v} for a, qs in EXPECTED.items() for q, v in qs.items()])
df = df_jev.merge(df_gem, on=["answer", "question_id"], how="left").merge(exp, on=["answer", "question_id"], how="left")
df["agree"] = df.jev_pred == df.gemini_pred
df["jev_correct"] = df.jev_pred == df.expected
df["gemini_correct"] = df.gemini_pred == df.expected
df.to_csv(OUT_DIR / "merged.csv", index=False)

active = df[~df.skipped]
print(f"Jev-Gemini 일치: {active.agree.sum()}/{len(active)}")
labeled = active[active.expected.notna()]
print(f"라벨 일치  Jev {labeled.jev_correct.sum()}/{len(labeled)}  Gemini {labeled.gemini_correct.sum()}/{len(labeled)}")

cols = ["answer", "question_id", "jev_pred", "jev_conf", "gemini_pred", "expected", "evidence", "reasoning"]
print("\n에스컬레이션 항목")
display(df[df.escalate][cols])
print("conf < 0.9 구간")
display(df[df.wide & ~df.escalate][cols])
print("Jev-Gemini 불일치 (전체)")
display(active[~active.agree][cols])

## 하이브리드 라우팅
CLAUDE.md 파이프라인 규칙 (코드로 분기):
- `auto`: Jev 확신도 높음 → Jev 판단을 추천
- `llm`: 에스컬레이션 + Jev와 Gemini 일치 → Gemini 근거와 함께 추천
- `human`: 에스컬레이션 + 불일치 → 채점자 직접 판단 (추천 없음)

In [ ]:
def route(r) -> str:
    if r.skipped:
        return "skip"
    if r.auto:
        return "auto"
    return "llm" if r.agree else "human"

df["route"] = df.apply(route, axis=1)
df["hybrid_pred"] = df.apply(lambda r: r.jev_pred if r.route in ("auto", "llm") else None, axis=1)
display(df.route.value_counts())
display(df[df.route.isin(["llm", "human"])][["answer", "question_id", "route", "jev_pred", "jev_conf", "gemini_pred", "expected", "reasoning"]])

## 하이브리드 vs Gemini 단독
- 라벨 기준은 skip이 아니고 라벨이 있는 칸
- 비용: Gemini는 답안당 1회 호출. 하이브리드는 에스컬레이션 항목이 있는 답안에서만 호출한다고 보고, 이번 호출(질문 전체 포함)을 상한으로 계산
- 지연: 답안별로 Jev 시간 + (에스컬레이션 항목이 있는 답안만) Gemini 시간

In [ ]:
lab = df[~df.skipped & df.expected.notna()]
rec = lab[lab.route != "human"]

calls = gem_calls.set_index("answer")
call_cost = (calls.input_tokens * GEMINI_PRICE["input"] + (calls.output_tokens + calls.thought_tokens) * GEMINI_PRICE["output"]) / 1e6
escalated_answers = df[df.escalate].answer.unique()
jev_cost = df_jev.groupby("answer").jev_input_tokens.first() * JEV_PRICE_INPUT / 1e6
jev_latency = df_jev.groupby("answer").jev_latency.first()
hybrid_latency = jev_latency + calls.gemini_latency.where(calls.index.isin(escalated_answers), 0)

summary = pd.DataFrame({
    "hybrid": {
        "추천 칸": f"{len(rec)}/{len(lab)}",
        "채점자 직접 판단": int((lab.route == "human").sum()),
        "추천 중 라벨 일치": f"{(rec.hybrid_pred == rec.expected).sum()}/{len(rec)}",
        "비용 USD (6답안)": round(jev_cost.sum() + call_cost[call_cost.index.isin(escalated_answers)].sum(), 5),
        "답안당 지연 평균 s": round(hybrid_latency.mean(), 2),
        "답안당 지연 최대 s": round(hybrid_latency.max(), 2),
    },
    "gemini_only": {
        "추천 칸": f"{len(lab)}/{len(lab)}",
        "채점자 직접 판단": 0,
        "추천 중 라벨 일치": f"{lab.gemini_correct.sum()}/{len(lab)}",
        "비용 USD (6답안)": round(call_cost.sum(), 5),
        "답안당 지연 평균 s": round(calls.gemini_latency.mean(), 2),
        "답안당 지연 최대 s": round(calls.gemini_latency.max(), 2),
    },
})
summary

## thinking_level 비교
02 첫 실행에서 thinking 토큰(호출당 761~1443)이 출력 토큰보다 많아 비용·지연의 절반 이상을 차지. `low`로 줄였을 때 판단 품질이 유지되는지 확인.

`app.judges.gemini.GeminiJudge` 사용 (타입별 스키마 + 1회 재시도 포함). 기본값 결과는 위 `df_gem`. 호출 6회 이상.

In [ ]:
from app.judges.gemini import GeminiJudge

judge_low = GeminiJudge(thinking_level="low")
low_rows, low_calls = [], []
for name, text in ANSWERS.items():
    for j in judge_low.evaluate(PROBLEM, text, QUESTIONS):
        low_rows.append({"answer": name, "question_id": j.question_id, "low_pred": j.pred, "low_reasoning": j.reasoning})
    low_calls.append({"answer": name, "latency": judge_low.last_latency, **judge_low.last_usage})

df_low = pd.DataFrame(low_rows)
low_calls = pd.DataFrame(low_calls).set_index("answer")
cmp_low = df.merge(df_low, on=["answer", "question_id"])
lab_low = cmp_low[~cmp_low.skipped & cmp_low.expected.notna()]

print(f"라벨 일치  default {lab_low.gemini_correct.sum()}/{len(lab_low)}  low {(lab_low.low_pred == lab_low.expected).sum()}/{len(lab_low)}")
print(f"default와 판단 일치 {(cmp_low.low_pred == cmp_low.gemini_pred).sum()}/{len(cmp_low)}, null {cmp_low.low_pred.isna().sum()}")

cost = lambda c: (c.input_tokens * GEMINI_PRICE["input"] + (c.output_tokens + c.thought_tokens) * GEMINI_PRICE["output"]) / 1e6
display(pd.DataFrame({
    "default": {"latency_mean": calls.gemini_latency.mean(), "thought_tokens_mean": calls.thought_tokens.mean(), "cost_per_call": cost(calls).mean()},
    "low": {"latency_mean": low_calls.latency.mean(), "thought_tokens_mean": low_calls.thought_tokens.mean(), "cost_per_call": cost(low_calls).mean()},
}).round(5))
display(cmp_low[cmp_low.low_pred != cmp_low.gemini_pred][["answer", "question_id", "gemini_pred", "low_pred", "expected", "low_reasoning"]])

## 모듈 확인: `app.judges.routing.grade`
노트북에서 검증한 흐름을 앱 모듈로 다시 실행. 에스컬레이션 항목만 Gemini로 보내므로 호출은 에스컬레이션이 있는 답안 수만큼.
위 `route` 결과와 같아야 함 (Jev·Gemini 응답이 실행마다 조금씩 달라 경계 칸은 바뀔 수 있음).

In [ ]:
from app.judges.jev import JevJudge
from app.judges.routing import grade

primary, secondary = JevJudge(jev), GeminiJudge()
mod_rows = []
for name, text in ANSWERS.items():
    for r in grade(PROBLEM, text, QUESTIONS, primary, secondary):
        mod_rows.append({"answer": name, "question_id": r.question_id, "module_route": str(r.route), "module_pred": r.pred, "reasoning": r.reasoning})

df_mod = pd.DataFrame(mod_rows).merge(df[["answer", "question_id", "route", "expected"]], on=["answer", "question_id"])
display(df_mod.module_route.value_counts())
print(f"노트북 route와 일치 {(df_mod.module_route == df_mod.route).sum()}/{len(df_mod)}")
display(df_mod[df_mod.module_route.isin(["llm", "human"]) | (df_mod.module_route != df_mod.route)])

## 관찰

> 문항 1개, 답안 6개 기준. 라벨은 Claude 초안이라 독립 라벨이 아니며, LLM이 매긴 라벨은 LLM 판단과 닮기 쉬워 Gemini 단독에 유리하게 작용할 수 있음.

### 1. 에스컬레이션 (첫 실행)
- 에스컬레이션 2칸: `partial` misc_is_optimizer(conf .54), `thin` mentions_weight_update(conf .54)
- 두 칸 모두 Gemini 판단이 Jev·라벨과 일치 → `llm` 경로. `human` 경로는 0건이라 **불일치 처리는 미검증** (평가셋의 경계 사례에서 확인 필요)
- 라우팅: auto 34, llm 2, skip 6

### 2. 근거 인용
- 근거가 있는 15/42칸 모두 원문 그대로 인용 (환각 0)
- 근거 없는 27칸은 전부 "언급 없음"으로 False 판단한 칸 → 빈 리스트가 맞는 처리
- 그래도 앱에서는 원문에 없는 인용을 버리는 검증을 유지 (`verify_evidence`)

### 3. 하이브리드 vs Gemini 단독
- 라벨 일치: Jev 36/36, Gemini 단독 35/36 (1건은 판단이 아니라 스키마 위반)
- 이 데이터에서 하이브리드의 이점은 정확도가 아니라 속도·비용: Jev 0.18s vs Gemini 6~10s, 하이브리드는 에스컬레이션 있는 답안만 Gemini 호출
- 비용(6답안): 하이브리드 상한 $0.013, Gemini 단독 $0.040
- 재실행: 라벨 일치 둘 다 36/36. 답안당 지연 평균 하이브리드 4.1s vs 단독 8.0s (최대는 둘 다 약 12s), 비용 $0.017 vs $0.035

### 4. Gemini 응답 품질
- 스키마 위반 1건: `partial` mentions_weight_update(noul)에 `verdict` 대신 `level: 2`. 단건 실행에서는 정상 → 비결정적 형식 오류
  - 대응: 타입별 스키마(`type` Literal + 필수 필드) + 형식이 틀린 질문만 1회 재시도, 그래도 실패하면 `pred=None` → `human` (`app/judges/gemini.py`)
- reasoning: 한국어, 판단과 일관, 기준 문구를 정확히 짚음
- thinking 토큰이 호출당 761~1443으로 출력보다 많음 → thinking_level 비교:
  - `low`: thought 토큰 893 → 0, 지연 8.0s → 3.7s, 호출당 비용 $0.0058 → $0.0027
  - 판단: default와 41/42 일치, 라벨 35/36. 틀린 1칸은 `misconception` mentions_weight_update ("오차를 바탕으로 가중치를 고친다"를 기울기 기반 갱신으로 관대하게 해석) → 01부터 가장 애매했던 경계 칸
  - **결정: 기본값 유지.** 에스컬레이션은 애매한 칸만 받으므로 `low`가 놓친 바로 그 유형이 주 입력. 호출 수가 적어 비용 차이도 작음

### 설계 반영 사항
- `Judge` Protocol: `evaluate(problem, answer, questions) -> list[Judgment]` (`app/judges/base.py`)
- `Judgment` 공통 필드: `question_id`, `type`, `pred`, `conf`(Jev만), `raw`, `evidence`, `reasoning`, `skipped`
- `GeminiJudge`: Jev 결과를 보지 않음, 에스컬레이션 질문만 받음, `store=False`
- 라우팅(`app/judges/routing.py`): skip / auto / llm(일치) / human(불일치 또는 LLM 실패). `human`에도 LLM 근거는 참고용으로 전달
- 무료 등급은 합성 데이터만. 실제 학생 답안은 결제 계정이 연결된 프로젝트에서